# SCENIC+ eGRN inference is a Snakemake pipeline — Patient 1 (03H096 / PB2)

This folder has two different kinds of files. Do not mix them up.

| File | What it is |
|---|---|
| `SCENICPLUS_preprocessing_Patient1_03H096.ipynb` | Jupyter notebook. Builds peaks, cisTopic and `region_sets/`. **Cluster job.** |
| `config.yaml` | **The SCENIC+ Snakemake config.** This is what actually infers the eGRNs. Not a notebook. |
| this notebook | Only documents how that Snakemake job was submitted. |

**The Jupyter notebook does not perform eGRN inference; inference is run through the SCENIC+ Snakemake workflow.** SCENIC+ ships a Snakemake workflow (`Snakefile`). After preprocessing finishes, you initialise that workflow, drop in `config.yaml`, and run `snakemake` on a **compute cluster**.

Do **not** run preprocessing or Snakemake on a laptop. Mallet LDA, MACS2, motif enrichment and GBM linking need cluster CPUs and memory. This job used **20 cores** and HTCondor node scratch (`temp_dir: ${_CONDOR_SCRATCH_DIR}`).

Official docs: https://scenicplus.readthedocs.io/en/latest/human_cerebellum.html


## What the Snakemake pipeline does

Snakemake reads `config.yaml` and runs the SCENIC+ rules in order:

1. Combine RNA (`scRNA/adata.h5ad`) with cisTopic accessibility
2. Motif enrichment (cisTarget + DEM) using the hg38 motif databases
3. Build cistromes (TF–region)
4. Link TFs to genes and regions to genes (GBM)
5. Assemble eRegulons
6. Score eRegulon activity with AUCell
7. Write `outs/scplusmdata.h5mu` (the object used for Figure 4)

The `Snakefile` is **not** in this GitHub repo. It is generated from the `scenicplus` Python package.


## Cluster commands (exact)

Preprocessing must already have written:

- `outs/cistopic_obj.pkl`
- `scRNA/adata.h5ad`
- `outs/region_sets/`

Then, on the cluster, in the patient folder:


In [ ]:
# 1. Create the official SCENIC+ Snakemake folder (Snakefile + a default config.yaml)
mkdir -p scplus_pipeline
scenicplus init_snakemake --out_dir scplus_pipeline

# That creates:
#   scplus_pipeline/Snakemake/workflow/Snakefile
#   scplus_pipeline/Snakemake/config/config.yaml   <- default; replace it next


In [ ]:
# 2. Replace the default config with the Patient 1 config from this repository
cp PATH_TO_REPO/05_Gene_regulatory_networks/config.yaml \
   scplus_pipeline/Snakemake/config/config.yaml

# Set PATH_TO_PROJECT and PATH_TO_MOTIF_DB in that copy if you are not
# on the original cluster. Do not change the parameters.


In [ ]:
# 3. Submit as a cluster job (20 cores, matching n_cpu in the YAML)
cd scplus_pipeline/Snakemake
snakemake --cores 20


## The full `config.yaml` that was submitted

This is the file in this folder (`05_Gene_regulatory_networks/config.yaml`). Paths are the cluster paths from the Patient 1 (PB2) run.

- `input_data` — cisTopic object, RNA AnnData, region-set BEDs, motif databases
- `output_data` — everything Snakemake writes, including `scplusmdata.h5mu`
- `params_general` — 20 CPUs; HTCondor scratch for temp files
- `params_data_preparation` / `params_motif_enrichment` / `params_inference` — SCENIC+ defaults used for the paper


```yaml
# SCENIC+ Snakemake config.yaml — Patient 1 (03H096 / PB2)
# Replace PATH_TO_PROJECT (patient folder with scRNA/ and outs/) and PATH_TO_MOTIF_DB.
# Submit on a cluster: scenicplus init_snakemake, copy this file to
# Snakemake/config/config.yaml, then snakemake --cores 20.

input_data:
  cisTopic_obj_fname: "PATH_TO_PROJECT/outs/cistopic_obj.pkl"
  GEX_anndata_fname: "PATH_TO_PROJECT/scRNA/adata.h5ad"
  region_set_folder: "PATH_TO_PROJECT/outs/region_sets"
  ctx_db_fname: "PATH_TO_MOTIF_DB/hg38_screen_v10_clust.regions_vs_motifs.rankings.feather"
  dem_db_fname: "PATH_TO_MOTIF_DB/hg38_screen_v10_clust.regions_vs_motifs.scores.feather"
  path_to_motif_annotations: "PATH_TO_MOTIF_DB/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl"


output_data:
  # output for prepare_GEX_ACC .h5mu
  combined_GEX_ACC_mudata: "PATH_TO_PROJECT/outs/ACC_GEX.h5mu"
  # output for motif enrichment results .hdf5
  dem_result_fname: "PATH_TO_PROJECT/outs/dem_results.hdf5"
  ctx_result_fname: "PATH_TO_PROJECT/outs/ctx_results.hdf5"
  # output html for motif enrichment results .html
  output_fname_dem_html: "PATH_TO_PROJECT/outs/dem_results.html"
  output_fname_ctx_html: "PATH_TO_PROJECT/outs/ctx_results.html"
  # output for prepare_menr .h5ad
  cistromes_direct: "PATH_TO_PROJECT/outs/cistromes_direct.h5ad"
  cistromes_extended: "PATH_TO_PROJECT/outs/cistromes_extended.h5ad"
  # output tf names .txt
  tf_names: "PATH_TO_PROJECT/outs/tf_names.txt"
  # output for download_genome_annotations .tsv
  genome_annotation: "PATH_TO_PROJECT/outs/genome_annotation.tsv"
  chromsizes: "PATH_TO_PROJECT/outs/chromsizes.tsv"
  # output for search_space .tsb
  search_space: "PATH_TO_PROJECT/outs/search_space.tsv"
  # output tf_to_gene .tsv
  tf_to_gene_adjacencies: "PATH_TO_PROJECT/outs/tf_to_gene_adj.tsv"
  # output region_to_gene .tsv
  region_to_gene_adjacencies: "PATH_TO_PROJECT/outs/region_to_gene_adj.tsv"
  # output eGRN .tsv
  eRegulons_direct: "PATH_TO_PROJECT/outs/eRegulon_direct.tsv"
  eRegulons_extended: "PATH_TO_PROJECT/outs/eRegulons_extended.tsv"
  # output AUCell .h5mu
  AUCell_direct: "PATH_TO_PROJECT/outs/AUCell_direct.h5mu"
  AUCell_extended: "PATH_TO_PROJECT/outs/AUCell_extended.h5mu"
  # output scplus mudata .h5mu
  scplus_mdata: "PATH_TO_PROJECT/outs/scplusmdata.h5mu"


params_general:
  temp_dir: "${_CONDOR_SCRATCH_DIR}"
  n_cpu: 20
  seed: 666

params_data_preparation:
  # Params for prepare_GEX_ACC
  bc_transform_func: "\"lambda x: f'{x}'\""
  is_multiome: True
  key_to_group_by: ""
  nr_cells_per_metacells: 10
  # Params for prepare_menr
  direct_annotation: "Direct_annot"
  extended_annotation: "Orthology_annot"
  # Params for download_genome_annotations
  species: "hsapiens"
  biomart_host: "http://www.ensembl.org"
  # Params for search_space
  search_space_upstream: "1000 150000"
  search_space_downstream: "1000 150000"
  search_space_extend_tss: "10 10"

params_motif_enrichment:
  species: "homo_sapiens"
  annotation_version: "v10nr_clust"
  motif_similarity_fdr: 0.001
  orthologous_identity_threshold: 0.0
  annotations_to_use: "Direct_annot Orthology_annot"
  fraction_overlap_w_dem_database: 0.4
  dem_max_bg_regions: 500
  dem_balance_number_of_promoters: True
  dem_promoter_space: 1_000
  dem_adj_pval_thr: 0.05
  dem_log2fc_thr: 1.0
  dem_mean_fg_thr: 0.0
  dem_motif_hit_thr: 3.0
  fraction_overlap_w_ctx_database: 0.4
  ctx_auc_threshold: 0.005
  ctx_nes_threshold: 3.0
  ctx_rank_threshold: 0.05

params_inference:
  # Params for tf_to_gene
  tf_to_gene_importance_method: "GBM"
  # Params regions_to_gene
  region_to_gene_importance_method: "GBM"
  region_to_gene_correlation_method: "SR"
  # Params for eGRN inference
  order_regions_to_genes_by: "importance"
  order_TFs_to_genes_by: "importance"
  gsea_n_perm: 1000
  quantile_thresholds_region_to_gene: "0.85 0.90 0.95"
  top_n_regionTogenes_per_gene: "5 10 15"
  top_n_regionTogenes_per_region: ""
  min_regions_per_gene: 0
  rho_threshold: 0.05
  min_target_genes: 3
```
